In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.11.2
numpy 2.5.3
pandas 3.0.6
sklearn 1.9.1
torch 2.14.0
cpu


# 准备数据集

In [5]:
"""
为什么不能直接放 data 文件夹
默认位置固定：fetch_california_housing()默认从～/scikit_learn_data/读取数据，这是scikit-learn约定的全局缓存位置，不会自动去你的项目的 data/ 里找
文件格式要求：它期望的缓存文件是 cal_housing.pkz，而不是你从网站下载的.tgz压缩包。程序内部会把.tgz解压、调整列顺序，再用joblib压缩成.pkz格式存起来
"""
from sklearn.datasets import get_data_home
print(get_data_home())

/Users/qingjiabu/scikit_learn_data


In [6]:
"""
此处是将 /Users/qingjiabu/scikit_learn_data/cal_housing.tgz 文件转变为 pkz 类型的文件

替代方案：
    # 假设你的项目里有个 data 文件夹，里面有 cal_housing.pkz 文件
    housing = fetch_california_housing(data_home='./data')
"""
import numpy as np
import os
import tarfile
import joblib
from sklearn.datasets._base import _pkl_filepath, get_data_home

# 你的 tgz 文件路径（如果已经放进 scikit_learn_data 目录，就用那个路径）
archive_path = "cal_housing.tgz"

data_home = get_data_home(data_home=None)
if not os.path.exists(data_home):
    os.makedirs(data_home)

filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    # 列的顺序要和 scikit-learn 期望的一致
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]
    joblib.dump(cal_housing, filepath, compress=6)

print("转换完成，缓存文件已生成:", filepath)

FileNotFoundError: [Errno 2] No such file or directory: 'cal_housing.tgz'

## 这里才是正式数据集准备

In [7]:
from sklearn.datasets import fetch_california_housing

In [8]:
# 默认位置固定：fetch_california_housing()默认从～/scikit_learn_data/读取数据，这是scikit-learn约定的全局缓存位置，不会自动去你的项目的 data/ 里找
housing = fetch_california_housing()
print(housing.DESCR)
"""
(20640, 8)：加利福利亚房价数据集总共有20640个样本，8个特征
特征：
    longitude、latitude、housingMedianAge、totalRooms、totalBedrooms、population、households、medianIncome
target：
    medianHouseValue
"""
print(housing.data.shape)
# target总共有20640个是因为有多少个样本就有多少个target标签
print(housing.target.shape)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib:
https://lib.stat.cmu.edu/datasets/houses.zip

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block g

In [11]:
print(type(housing.data))
print(type(housing.target))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [9]:
import pprint  #打印的格式比较 好看（pprint: pretty print）

pprint.pprint(housing.data[0:5])
print('-'*50)
pprint.pprint(housing.target[0:5])

array([[ 8.32520000e+00,  4.10000000e+01,  6.98412698e+00,
         1.02380952e+00,  3.22000000e+02,  2.55555556e+00,
         3.78800000e+01, -1.22230000e+02],
       [ 8.30140000e+00,  2.10000000e+01,  6.23813708e+00,
         9.71880492e-01,  2.40100000e+03,  2.10984183e+00,
         3.78600000e+01, -1.22220000e+02],
       [ 7.25740000e+00,  5.20000000e+01,  8.28813559e+00,
         1.07344633e+00,  4.96000000e+02,  2.80225989e+00,
         3.78500000e+01, -1.22240000e+02],
       [ 5.64310000e+00,  5.20000000e+01,  5.81735160e+00,
         1.07305936e+00,  5.58000000e+02,  2.54794521e+00,
         3.78500000e+01, -1.22250000e+02],
       [ 3.84620000e+00,  5.20000000e+01,  6.28185328e+00,
         1.08108108e+00,  5.65000000e+02,  2.18146718e+00,
         3.78500000e+01, -1.22250000e+02]])
--------------------------------------------------
array([4.526, 3.585, 3.521, 3.413, 3.422])


In [12]:
from sklearn.model_selection import train_test_split

# 将数据集拆分为训练集和测试集
x_train_all, x_test, y_train_all, y_test = train_test_split(
    housing.data, housing.target, random_state=7
)
# 将训练集拆分成训练集和验证集
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_all, y_train_all, random_state=11
)
# 训练集
print(x_train.shape, y_train.shape)
# 验证集
print(x_valid.shape, y_valid.shape)
# 测试集
print(x_test.shape, y_test.shape)

dataset_maps = {
    "train": [x_train, y_train],
    "valid": [x_valid, y_valid],
    "test": [x_test, y_test],
}

(11610, 8) (11610,)
(3870, 8) (3870,)
(5160, 8) (5160,)


In [13]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
"""
StandardScaler()是一个类，创建了一个实例对象 scaler，一个空的“标准化器”，里面什么都没存
scaler.fit(x_train)：根据训练集计算每个特征的均值和标准差，存进scaler对象内部
"""
scaler = StandardScaler()
"""
把结果存到对象属性里：
    scaler.mean_    # 形状 (D,)，每个特征的均值
    scaler.scale_   # 形状 (D,)，每个特征的标准差
    scaler.var_     # 形状 (D,)，每个特征的方差
    scaler.n_features_in_   # 特征数 D
    scaler.n_samples_seen_  # 样本数 N
fit只计算并存储，不改变x_train
"""
scaler.fit(x_train)

StandardScaler()

## 构建数据集

In [14]:
"""
常见dunder方法：
    __init__           创建对象时                obj = MyClass()
    __str__            str(obj)/print(obj)      打印对象
    __len__             len(obj)                求长度
    __getitem__         obj[key]                索引访问
    __iter__            for x in obj            迭代
    __call__            obj()                   把对象当函数调用
    __name__            模块名                  if __name__=='__main__':
"""
from torch.utils.data import Dataset

class HousingDataset(Dataset):
    """
    方法名左右各有两个下划线：这叫dunder方法（dunder = double underscore，双下划线），中文常叫魔法方法或特殊方法
    他们是Python内置约定的特殊方法，不是普通方法。一般不直接调用他们，而是由Python在特殊操作时自动调用
    
    默认 mode 是 训练集train
    """
    def __init__(self, mode='train'):
        # 解包操作，key、value
        self.x, self.y = dataset_maps[mode]
        # 把NumPy数组先用scaler标准化，再转成PyTorch张量，最后转成float32类型，赋值回self.x
        # 通过fit之后标准化器中的均值和方差标准化特征x中的数据
        self.x = torch.from_numpy(scaler.transform(self.x)).float()
        self.y = torch.from_numpy(self.y).float().reshape(-1, 1)
            
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
# 实例对象数据集
train_ds = HousingDataset("train")
valid_ds = HousingDataset("valid")
test_ds = HousingDataset("test")

In [15]:
train_ds[1]

(tensor([-0.2981,  0.3523, -0.1092, -0.2506, -0.0341, -0.0060,  1.0806, -1.0611]),
 tensor([1.5140]))

### DataLoader

In [ ]:
from torch.utils.data import DataLoader

batch_size = 16
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

# 定义模型

In [ ]:
#回归模型我们只需要1个数
"""
model.named_parameters()
│
├── model._parameters（当前模块自己的参数） → 空（MyNet 自己没直接定义参数）
│
└── 遍历 model._modules:                                                  （当前模块的子模块）
    └── "linear_relu_stack" → Sequential
        │
        ├── Sequential._parameters → 空
        │
        └── 遍历 Sequential._modules:
            ├── "0" → Linear(input_dim, 30)
            │     └── Linear._parameters:
            │           "weight" → [30, input_dim]
            │           "bias"   → [30]
            │
            ├── "1" → ReLU
            │     └── ReLU._parameters → 空（无参数）
            │
            └── "2" → Linear(30, 1)
                  └── Linear._parameters:
                        "weight" → [1, 30]
                        "bias"   → [1]
"""
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        """
        class Sequential(Module): -> 这里继承了Module类
        
        整体调用链：
        model(x)
          │
          ▼
        __call__(x)
          │
          ▼
        _wrapped_call_impl(x)
          │
          ├── _compiled_call_impl 不为 None？
          │     ├── 是 → 走编译版（torch.compile 优化过）
          │     └── 否 → _call_impl(x)
          │                │
          │                ▼
          │             处理 hook
          │                │
          │                ▼
          │             forward(x)
          │
          ▼
        输出
                      
        _compiled_call_impl(x)：默认是None；只用当你用了torch.compile(model)之后，它才会被赋值；赋的值是一个编译优化过的调用函数
        _call_impl才是真正处理hook和forward的地方：
            def _call_impl(self, *args, **kwargs):
                # 如果有 forward hook / backward hook
                # 走带 hook 的路径
                # 否则直接调 forward
                return self.forward(*args, **kwargs)
        """
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_dim, 30),
            nn.ReLU(),
            nn.Linear(30, 1)
            )
        """
        里面的 nn.Linear(...)、nn.ReLU() 是注册到该属性内部的子模块，不是 self 直接持有的实例属性
        
        _modules: dict[str, Module]：
            _modules：                                   属性名
            : dict[str, Module]                         类型注解：一个字典，键是字符串，值是Module
            这里只是声明类型，没有赋值。在类定义里，这种只有注解、没有赋值的语句不会创建类属性
            
        传入普通模块：
            # container.py
            for idx, module in enumerate(args):
                self.add_module(str(idx), module)                   # 用索引的字符串作模块名
            为什么用str(idx) -> add_module要求名字是字符串，而idx是整数，所以str(idx)转换
            self.add_module(name, module):
                把module注册到self._modules（一个OrderedDict）
                之后named_parameters()、.to(device)、state_dict()都能识别它
        
        # module.py
        def add_module(self, name: str, module: Optional["Module"]) -> None:
            for hook in _global_module_registration_hooks.values():
                output = hook(self, name, module)
                if output is not None:
                    module = output
            self._modules[name] = module
        """
    def forward(self, x):
        # x.shape [batch size, 8]
        logits = self.linear_relu_stack(x)
        # logits.shape [batch size, 1]
        return logits

In [ ]:
"""
源码有点难啃，等后面回头的时候再看吧！
    1. 把module.py里 def __setattr__(self, name: str, value: Union[Tensor, "Module"]) -> None: 方法里的代码分析一下
    2. _modules是如何存子模块的，是只要是属性就会都存起来吗
"""

In [ ]:
"""
这是一个早停(Early Stopping)回调类，用来验证指标不再提升时提前终止训练，避免过拟合、节省时间
整体思路：
    训练时每个epoch结束记录一个监控指标（如验证准确率或验证损失）：
        指标持续提升 -> 继续训练，重置计数器
        指标连续patience轮没有明显提升 -> 触发早停
        
__init__：初始化状态
    patience：容忍多少轮“没提升”后停止，默认5
    min_delta：最小提升幅度，小于它不算提升，默认0.01
    best_metric：历史最佳指标，初始为-1
    counter：连续无提升的轮数
    注意：best_metric = -1：这是一个“初始占位值”。假设监控的是准确率（0-1之间），任何真实准确率都大于-1，所以第一轮一定会更新best_metric。但如果监控的是损失（越小越好），这个设计就有问题。
"""
class EarlyStopCallback:
    def __init__(self, patience=5, min_delta=0.01):
        """

        Args:
            patience (int, optional): Number of epochs with no improvement after which training will be stopped.. Defaults to 5.
            min_delta (float, optional): Minimum change in the monitored quantity to qualify as an improvement, i.e. an absolute 
                change of less than min_delta, will count as no improvement. Defaults to 0.01.
        """
        # patience：容忍多少轮“没提升”后停止，默认5
        self.patience = patience
        # min_delta：最小提升幅度，小于它不算提升，默认0.01
        self.min_delta = min_delta
        # best_metric：历史最佳指标，初始为-1
        self.best_metric = -1
        # counter：连续无提升的轮数
        self.counter = 0
        
    def __call__(self, metric):
        # 为什么要min_delta：避免“微小波动”被误判为提升。比如准确率从0.9000到0.9001，几乎没变，不应该重置计数器。只有提升超过min_delta才算数
        if metric >= self.best_metric + self.min_delta:
            # update best metric
            self.best_metric = metric
            # reset counter 
            self.counter = 0
        else: 
            self.counter += 1
          
    """
    用@property装饰，像属性一样访问：callback.early_stop，不用加括号
    连续无提升轮数达到patience -> 返回True -> 训练停止
    """  
    @property
    def early_stop(self):
        return self.counter >= self.patience


In [ ]:
from sklearn.metrics import accuracy_score

@torch.no_grad()
def evaluating(model, dataloader, loss_fct):
    loss_list = []
    for datas, labels in dataloader:
        datas = datas.to(device)
        labels = labels.to(device)
        # 前向计算
        logits = model(datas)
        loss = loss_fct(logits, labels)         # 验证集损失
        loss_list.append(loss.item())
        
    return np.mean(loss_list)

In [ ]:
# 自定义MSEloss

def mse_loss(array1, array2):
    return ((array1 - array2) ** 2).mean()

In [ ]:
# 假设 y_true 是真实值的张量，y_pred 是模型预测值的张量
y_true = torch.tensor([3, -0.5, 2, 7])
y_pred = torch.tensor([2.5, 0.0, 2.0, 8])

# 计算 MSE 损失
loss = F.mse_loss(y_pred, y_true)
loss

In [ ]:
mse_loss(y_pred, y_true)

In [ ]:
# 训练
def training(
    model, 
    train_loader, 
    val_loader, 
    epoch, 
    loss_fct, 
    optimizer, 
    tensorboard_callback=None,
    save_ckpt_callback=None,
    early_stop_callback=None,
    eval_step=500,
    ):
    record_dict = {
        "train": [],
        "val": []
    }
    
    global_step = 0
    model.train()
    with tqdm(total=epoch * len(train_loader)) as pbar:
        for epoch_id in range(epoch):
            # training
            for datas, labels in train_loader:
                datas = datas.to(device)
                labels = labels.to(device)
                # 梯度清空
                optimizer.zero_grad()
                # 模型前向计算
                logits = model(datas)
                # 计算损失
                loss = loss_fct(logits, labels)
                # 梯度回传
                loss.backward()
                # 调整优化器，包括学习率的变动等
                optimizer.step()
 
                loss = loss.cpu().item()
                # record
                
                record_dict["train"].append({
                    "loss": loss, "step": global_step
                })
                
                # evaluating
                if global_step % eval_step == 0:
                    model.eval()
                    val_loss = evaluating(model, val_loader, loss_fct)
                    record_dict["val"].append({
                        "loss": val_loss, "step": global_step
                    })
                    model.train()

                    # 早停 Early Stop
                    if early_stop_callback is not None:
                        early_stop_callback(-val_loss)
                        if early_stop_callback.early_stop:
                            print(f"Early stop at epoch {epoch_id} / global_step {global_step}")
                            return record_dict
                    
                # update step
                global_step += 1
                pbar.update(1)
                pbar.set_postfix({"epoch": epoch_id})
        
    return record_dict

In [ ]:
epoch = 100

model = NeuralNetwork()

# 1. 定义损失函数 采用自定义的MSE
loss_fct = mse_loss
# loss_fct = F.mse_loss
# 2. 定义优化器 采用SGD
# Optimizers specified in the torch.optim package
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# 3. early stop
early_stop_callback = EarlyStopCallback(patience=10, min_delta=1e-3)

model = model.to(device)
record = training(
    model, 
    train_loader, 
    val_loader, 
    epoch, 
    loss_fct, 
    optimizer, 
    early_stop_callback=early_stop_callback,
    eval_step=len(train_loader)
    )

In [ ]:
#画线要注意的是损失是不一定在零到1之间的
def plot_learning_curves(record_dict, sample_step=500):
    # build DataFrame
    train_df = pd.DataFrame(record_dict["train"]).set_index("step").iloc[::sample_step]
    val_df = pd.DataFrame(record_dict["val"]).set_index("step")

    # plot
    for idx, item in enumerate(train_df.columns):
        plt.plot(train_df.index, train_df[item], label=f"train_{item}")
        plt.plot(val_df.index, val_df[item], label=f"val_{item}")
        plt.grid()
        plt.legend()
        # plt.xticks(range(0, train_df.index[-1], 10*sample_step), range(0, train_df.index[-1], 10*sample_step))
        plt.xlabel("step")

        plt.show()

plot_learning_curves(record)  #横坐标是 steps

# 测试集

In [ ]:
model.eval()
loss = evaluating(model, val_loader, loss_fct)
print(f"loss:     {loss:.4f}")